# Final Dashboard

Let's improve the application we started building in the first project. We will be plotting all the ski resorts in our dataset (countries in all continents). 

### Tab 1: Map of Skiing Hotspots based on filters

We still want our scatter_mapbox map of ski_resorts with density as total slopes, but modify the layout so all interactive components are in a column to the left of the map. This should include:
1. A slider that sets a maximum price
2. A checkbox to filter to resorts with summer skiing
3. A checkbox to filter to resorts with night skiing
4. A checkbox to filter to resorts with a snowpark

### Tab 2: Country report & Resort Report card

We want this tab to include a bar chart of the top 10 resorts by the user's selected metric. This layout will include three "columns":

1. A sidebar that allows users to first select a continent. Based on the selected continent, a second dropdown will populate with the countries in that continent, allowing users to select a country. Finally, users will use a dropdown menu to select the column they want to plot in our bar chart. 

2. A bar chart where the x-axis is resort (consider removing x-axis labels), and y-axis is the users selected metric. You should only plot the top 10 resorts by the selected metric. 

3. A Resort Report Card. A basic version of this would a handful of key metrics by country. If you're daring, use interactive filtering to display the resort name, its elevation rank, slope rank, price rank, and cannon rank (based off of the SnowCannons column. 

This is going to be a bit tricky, and ultimately you can make this dashboard however you want, but this should be a great way to put your skills to the test!



In [ ]:
from dash import Dash, dcc, html, dash_table
import dash_bootstrap_components as dbc
from dash.dependencies import Output, Input
from dash.exceptions import PreventUpdate
from dash_bootstrap_templates import load_figure_template

import plotly.express as px
import pandas as pd
import numpy as np

# =============================================================================
# DATA LOADING
# =============================================================================
resorts = (
    pd.read_csv(r"C:\Users\MilosIlic\OneDrive - Valcon Business Development A S\Data Plaftom (Power BI)\Python\Interactive Dashboards with Plotly & Dash\resources\Course_Materials\Data\Ski Resorts\resorts.csv", encoding="ISO-8859-1")
    .assign(
        country_elevation_rank=lambda x: x.groupby("Country", as_index=False)["Highest point"].rank(ascending=False),
        country_price_rank=lambda x: x.groupby("Country", as_index=False)["Price"].rank(ascending=False),
        country_slope_rank=lambda x: x.groupby("Country", as_index=False)["Total slopes"].rank(ascending=False),
        country_cannon_rank=lambda x: x.groupby("Country", as_index=False)["Snow cannons"].rank(ascending=False),
    ))

# =============================================================================
# APP INITIALIZATION
# =============================================================================
dbc_css = "https://cdn.jsdelivr.net/gh/AnnMarieW/dash-bootstrap-templates/dbc.min.css"
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP, dbc_css])
load_figure_template("bootstrap")

# =============================================================================
# TAB 1: MAP LAYOUT
# =============================================================================
tab1_content = dbc.Container([
    html.H1(id="map-title", style={"text-align": "center"}, className="my-3"),
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Markdown("**Price Limit**"),
                    dcc.Slider(
                        id='price-slider',
                        min=0,
                        max=150,
                        step=25,
                        value=150,
                        marks={i: f'${i}' for i in range(0, 175, 25)},
                        className="dbc mb-4"
                    ),
                    
                    dcc.Markdown("**Feature Preferences**"),
                    dcc.Checklist(
                        id='summer-ski-checklist',
                        options=[{"label": " Has Summer Skiing", "value": "Yes"}],
                        value=[],
                        className="mb-2"
                    ),
                    dcc.Checklist(
                        id='night-ski-checklist',
                        options=[{"label": " Has Night Skiing", "value": "Yes"}],
                        value=[],
                        className="mb-2"
                    ),
                    dcc.Checklist(
                        id='snow-park-checklist',
                        options=[{"label": " Has Snow Park", "value": "Yes"}],
                        value=[],
                        className="mb-2"
                    ),
                ])
            ])
        ], width=3),
        
        dbc.Col([
            dcc.Graph(id='resort-map', style={'height': '600px'})
        ], width=9)
    ])
], fluid=True)

# =============================================================================
# TAB 2: COUNTRY REPORT LAYOUT (IMPROVED)
# =============================================================================
tab2_content = dbc.Container([
    html.H1(id="country-title", style={"text-align": "center"}, className="my-3"),
    dbc.Row([
        # Column 1: Sidebar
        dbc.Col([
            dcc.Markdown("**Select A Continent:**"),
            dcc.Dropdown(
                id='continent-dropdown',
                options=[{'label': cont, 'value': cont} for cont in sorted(resorts['Continent'].dropna().unique())],
                value="Europe",
                className="dbc mb-3",
                clearable=False
            ),
            
            dcc.Markdown("**Select A Country:**"),
            dcc.Dropdown(
                id='country-dropdown',
                value="Norway",
                className="dbc mb-3",
                clearable=False
            ),
            
            dcc.Markdown("**Select A Metric to Plot:**"),
            dcc.Dropdown(
                id='metric-dropdown',
                options=[
                    {'label': 'Total Slopes', 'value': 'Total slopes'},
                    {'label': 'Highest Point', 'value': 'Highest point'},
                    {'label': 'Lowest Point', 'value': 'Lowest point'},
                    {'label': 'Price', 'value': 'Price'},
                    {'label': 'Snow Cannons', 'value': 'Snow cannons'},
                    {'label': 'Total Lifts', 'value': 'Total lifts'},
                ],
                value='Price',
                className="dbc mb-3",
                clearable=False
            ),
        ], width=3),
        
        # Column 2: Bar chart with INITIAL HOVER DATA
        dbc.Col([
            dcc.Graph(
                id='metric-bar',
                hoverData={'points': [{'customdata': [resorts.query("Country == 'Norway'").nlargest(1, 'Price')['Resort'].iloc[0]]}]},
                style={'height': '600px'}
            )
        ], width=6),
        
        # Column 3: Resort Report Card
        dbc.Col([
            dcc.Markdown("### Resort Report Card", className="text-center"),
            dbc.Card(id="resort-name", className="mb-3 text-center", 
                     style={"fontSize": 20, "fontWeight": "bold", "padding": "10px"}),
            dbc.Row([
                dbc.Col([
                    dbc.Card(id="elevation-kpi", className="mb-2", 
                             style={"padding": "10px", "backgroundColor": "#f8f9fa"}),
                    dbc.Card(id="price-kpi", className="mb-2",
                             style={"padding": "10px", "backgroundColor": "#f8f9fa"}),
                ]),
                dbc.Col([
                    dbc.Card(id="slope-kpi", className="mb-2",
                             style={"padding": "10px", "backgroundColor": "#f8f9fa"}),
                    dbc.Card(id="cannon-kpi", className="mb-2",
                             style={"padding": "10px", "backgroundColor": "#f8f9fa"}),
                ])
            ])
        ], width=3)
    ])
], fluid=True)

# =============================================================================
# MAIN LAYOUT
# =============================================================================
app.layout = dbc.Container([
    html.H1("Ski Resort Dashboard", className="text-center my-4"),
    
    dbc.Tabs([
        dbc.Tab(tab1_content, label="Map of Skiing Hotspots", tab_id="tab-1"),
        dbc.Tab(tab2_content, label="Country Report & Resort Report Card", tab_id="tab-2"),
    ], id="tabs", active_tab="tab-1")
], fluid=True)

# =============================================================================
# CALLBACKS
# =============================================================================

# TAB 1: Map callback
@app.callback(
    Output("map-title", "children"),
    Output("resort-map", "figure"),
    Input("price-slider", "value"),
    Input("summer-ski-checklist", "value"),
    Input("night-ski-checklist", "value"),
    Input("snow-park-checklist", "value")
)
def update_map(price, summer_ski, night_ski, snow_park):
    """
    EXPLANATION: Filters resorts based on price and feature checkboxes,
    then creates a density mapbox showing resort distribution
    """
    title = f"Resorts with a ticket price less than ${price}"
    
    # Filter by price
    df = resorts.loc[resorts["Price"] <= price]
    
    # Filter by summer skiing if checked
    if "Yes" in summer_ski:
        df = df.loc[df["Summer skiing"] == "Yes"]
    
    # Filter by night skiing if checked
    if "Yes" in night_ski:
        df = df.loc[df["Nightskiing"] == "Yes"]
    
    # Filter by snow park if checked
    if "Yes" in snow_park:
        df = df.loc[df["Snowparks"] == "Yes"]
    
    # Create density map
    fig = px.density_mapbox(
        df,
        lat="Latitude",
        lon="Longitude",
        z="Total slopes",
        hover_name="Resort",
        center={"lat": 45, "lon": -100},
        zoom=2.5,
        mapbox_style="open-street-map",
        color_continuous_scale="Blues",
        height=600
    )
    
    return title, fig


# TAB 2: Update country dropdown based on continent
@app.callback(
    Output("country-dropdown", "options"),
    Output("country-dropdown", "value"),
    Input("continent-dropdown", "value")
)
def update_country_dropdown(continent):
    """
    EXPLANATION: When continent changes, populate country dropdown
    with countries from that continent. Uses numpy sort for clean ordering.
    """
    if not continent:
        raise PreventUpdate
    
    # Get sorted list of countries in the selected continent
    countries = np.sort(resorts.query("Continent == @continent")["Country"].dropna().unique())
    options = [{'label': country, 'value': country} for country in countries]
    
    # Set first country as default
    value = countries[0] if len(countries) > 0 else None
    
    return options, value


# TAB 2: Update bar chart
@app.callback(
    Output("country-title", "children"),
    Output("metric-bar", "figure"),
    Input("country-dropdown", "value"),
    Input("metric-dropdown", "value")
)
def update_bar_chart(country, metric):
    """
    EXPLANATION: Creates bar chart of top resorts. Uses custom_data
    to pass resort name for hover interactions. Removes x-axis labels
    for cleaner appearance.
    """
    if not country or not metric:
        raise PreventUpdate
    
    title = f"Top Resorts in {country} by {metric}"
    
    # Filter and sort by selected metric
    df = resorts.query("Country == @country").sort_values(metric, ascending=False)
    
    # Create bar chart with custom_data for hover
    fig = px.bar(
        df,
        x="Resort",
        y=metric,
        custom_data=["Resort"],  # KEY: Passes resort name to hoverData
        color=metric,
        color_continuous_scale="Blues"
    )
    
    # Remove x-axis labels for cleaner look
    fig.update_xaxes(showticklabels=False)
    fig.update_layout(showlegend=False, height=600)
    
    return title, fig


# TAB 2: Update resort report card using HOVER
@app.callback(
    Output("resort-name", "children"),
    Output("elevation-kpi", "children"),
    Output("price-kpi", "children"),
    Output("slope-kpi", "children"),
    Output("cannon-kpi", "children"),
    Input("metric-bar", "hoverData")  # Using HOVER instead of CLICK
)
def update_resort_card(hoverData):
    """
    EXPLANATION: When user hovers over a bar, extract the resort name
    from custom_data and display its ranking information in KPI cards.
    This approach is more intuitive than clicking - just hover to explore!
    """
    if not hoverData:
        raise PreventUpdate
    
    # Extract resort name from hoverData
    resort = hoverData["points"][0]["customdata"][0]
    
    # Query the specific resort
    df = resorts.query("Resort == @resort")
    
    # Create KPI text for each metric
    elevation_rank = f"Elevation Rank: {int(df['country_elevation_rank'].iloc[0])}"
    price_rank = f"Price Rank: {int(df['country_price_rank'].iloc[0])}"
    slope_rank = f"Slope Rank: {int(df['country_slope_rank'].iloc[0])}"
    cannon_rank = f"Cannon Rank: {int(df['country_cannon_rank'].iloc[0])}"
    
    return resort, elevation_rank, price_rank, slope_rank, cannon_rank


# =============================================================================
# RUN APP
# =============================================================================
if __name__ == '__main__':
    app.run(debug=True, port=8055)

C:\Users\MilosIlic\AppData\Local\Temp\ipykernel_16904\1581758475.py:201: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\MilosIlic\AppData\Local\Temp\ipykernel_16904\1581758475.py:201: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\MilosIlic\AppData\Local\Temp\ipykernel_16904\1581758475.py:201: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\MilosIlic\AppData\Local\Temp\ipykernel_16904\1581758475.py:201: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\MilosIlic\AppData\Local\Temp\ipykernel_16904\1581758475.py:201: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_m

In [3]:
resorts.head()

,ID,Resort,Latitude,Longitude,Country,Continent,Price,Season,Highest point,Lowest point,...,Total lifts,Lift capacity,Child friendly,Snowparks,Nightskiing,Summer skiing,country_elevation_rank,country_price_rank,country_slope_rank,country_cannon_rank
0,1,Hemsedal,60.928244,8.383487,Norway,Europe,46,November - May,1450,620,...,21,22921,Yes,Yes,Yes,No,2.0,1.5,4.0,2.0
1,2,Geilosiden Geilo,60.534526,8.206372,Norway,Europe,44,November - April,1178,800,...,24,14225,Yes,Yes,Yes,No,4.0,6.0,6.5,4.0
2,3,Golm,47.057810,9.828167,Austria,Europe,48,December - April,2110,650,...,11,16240,Yes,No,No,No,37.0,31.5,63.0,45.0
3,4,Red Mountain Resort-Rossland,49.105520,-117.846280,Canada,North America,60,December - April,2075,1185,...,8,9200,Yes,Yes,Yes,No,13.0,13.0,5.0,15.0
4,5,Hafjell,61.230369,10.529014,Norway,Europe,45,November - April,1030,195,...,18,21060,Yes,Yes,Yes,No,9.0,3.5,3.0,3.0
